# 🧠 NS-ARC Object-Centric (Slotted) Scaling Experiment
This notebook utilizes the completely refactored 3-Phase Slot-JEPA architecture.
- **Phase 1:** Syntactic Rule Pretraining (RbJEPA)
- **Phase 2:** Unsupervised Semantic Generalization (Pure JEPA)
- **Phase 3:** Target Prediction (World Model)

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')  
import matplotlib.pyplot as plt
import pathlib
import random
import shutil

# --- Kaggle & Modules ---
KAGGLE_REPO_PATH = '/kaggle/working/Model-Jepa'
if os.path.exists(KAGGLE_REPO_PATH):
    if KAGGLE_REPO_PATH not in sys.path:
        sys.path.insert(0, KAGGLE_REPO_PATH)
else:
    sys.path.insert(0, os.path.abspath('.'))
    
# Device Config
if torch.cuda.is_available(): DEVICE = 'cuda'
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available(): DEVICE = 'mps'
else: DEVICE = 'cpu'
print(f'Using device: {DEVICE}')

In [ ]:
import wandb
WANDB_PROJECT = "NS-ARC-Slotted"
import os
wandb.login(key=os.environ.get('WANDB_API_KEY', ''))

from modules.encoders import SlotTransformerEncoder, SlotDecoder
from modules.predictors import SlotJEPAPredictor
from modules.rule_encoders import RuleEncoder
from modules.world_models import SlotWorldModel32
from arc_data.rearc_dataset import ReARCDataset
from analysis.plot_utils import plot_reconstruction_dashboard
from analysis.langevin_solver import LangevinGridSculptor

BASE_CFG = {
    'device': DEVICE,
    'input_dim': 64,  
    'in_channels': 1,
    'patch_size': 14,
    'hidden_dim': 256,
    'latent_dim': 128,
    'action_dim': 10,
    'num_slots': 16, 
    'slot_iters': 3,
    'encoder_layers': 4
}

MODELS = {
    'context_encoder': SlotTransformerEncoder(BASE_CFG).to(DEVICE),
    'target_encoder': SlotTransformerEncoder(BASE_CFG).to(DEVICE),
    'predictor': SlotJEPAPredictor(BASE_CFG).to(DEVICE),
    'rule_encoder': RuleEncoder(BASE_CFG).to(DEVICE),
    'world_model': SlotWorldModel32(BASE_CFG).to(DEVICE)
}
MODELS['target_encoder'].load_state_dict(MODELS['context_encoder'].state_dict())
for param in MODELS['target_encoder'].parameters():
    param.requires_grad = False
    
print("✅ Initiated Object-Centric Modules.")

In [ ]:
def train_epoch_jepa(phase, dataset, modules, cfg, n_batches, batch_size, beta=1.0, epoch=0):
    if phase == 'rbjepa' or phase == 'jepa':
        opt = torch.optim.AdamW(
            list(modules['context_encoder'].parameters()) + 
            list(modules['predictor'].parameters()) + 
            list(modules['rule_encoder'].parameters()), 
            lr=1e-3
        )
    elif phase == 'wm':
        opt = torch.optim.AdamW(list(modules['world_model'].parameters()) + list(modules['context_encoder'].parameters()), lr=1e-4)

    total_loss = 0
    for b in range(n_batches):
        batch = dataset.sample(batch_size)
        states = batch['state'].to(cfg['device'])

        if phase in ['rbjepa', 'jepa']:
            with torch.no_grad():
                z_t = modules['target_encoder']({'state': states})['latent']
            
            masks = (torch.rand_like(states) > 0.5).float()
            z_c = modules['context_encoder']({'state': states * masks})['latent']
            z_pred = modules['predictor'](z_c)
            
            loss_jepa = F.mse_loss(z_pred, z_t)
            
            loss_ebc = torch.tensor(0.0, device=cfg['device'])
            if phase == 'rbjepa' and 'rules' in batch:
                rules = batch['rules']
                z_rules_list = modules['rule_encoder'](rules) 
                
                num_valid = 0
                for i in range(len(rules)):
                    if len(rules[i]) > 0:
                        rule_cen = z_rules_list[i].mean(dim=0)
                        vis_cen = z_pred[i].mean(dim=0)
                        loss_ebc += F.mse_loss(vis_cen, rule_cen)
                        num_valid += 1
                if num_valid > 0: loss_ebc = loss_ebc / num_valid
                
            loss = loss_jepa + beta * loss_ebc
            
        elif phase == 'wm':
            z_start = modules['context_encoder']({'state': states})['latent'].detach()
            action = torch.randn(batch_size, cfg['action_dim'], device=cfg['device']) 
            out = modules['world_model']({'latent': z_start, 'action': action})
            
            with torch.no_grad():
                target_z = modules['target_encoder']({'state': batch['target_state'].to(cfg['device'])})['latent']
            
            loss_dict = modules['world_model'].loss({'target_latent': target_z, 'target_reward': batch['target_reward'].to(cfg['device'])}, out)
            loss = loss_dict['loss']

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for m in modules.values() for p in m.parameters() if getattr(p, 'requires_grad', False)], 1.0)
        opt.step()
        
        if phase in ['rbjepa', 'jepa']:
            modules['target_encoder'].update_ema(modules['context_encoder'], momentum=0.996)
            
        total_loss += loss.item()
        
    return total_loss / n_batches

In [ ]:
# Initialize W&B and Data
dataset = ReARCDataset(data_path='/kaggle/working/Model-Jepa/arc_data/re-arc')
wb_run = wandb.init(project=WANDB_PROJECT, name="3-Phase-Run", reinit=True)
sculptor = LangevinGridSculptor(MODELS['target_encoder'], DEVICE)

os.makedirs("evaluation_reports/plots", exist_ok=True)

# --- PHASE 1: Syntactic (RbJEPA) ---
print("Starting Phase 1: Syntactic Pretraining (RbJEPA)")
dataset.pairs = []
dataset._use_mock(3000) # Force highly diverse synthetic shapes
for epoch in range(50):
    loss = train_epoch_jepa('rbjepa', dataset, MODELS, BASE_CFG, 100, 16, beta=1.0, epoch=epoch)
    wb_run.log({"Phase_1/Loss": loss})
    print(f"Phase 1 Epoch {epoch} Loss: {loss:.4f}")
    
    # 10 Epoch Langevin Diagnostic
    if epoch % 10 == 0:
        b = dataset.sample(1)
        z_t = MODELS['target_encoder']({'state': b['state'].to(DEVICE)})['latent']
        grid = sculptor.sculpt(z_t[0:1], (1,1,30,30), steps=150)
        viz_path = f"evaluation_reports/plots/sculpt_p1_{epoch}.png"
        plot_reconstruction_dashboard(b['state'][0,0], grid[0,0], z_t[0], epoch, viz_path)
        wb_run.log({f"Phase_1/Sculpt": wandb.Image(viz_path)})

# --- PHASE 2: Semantic (JEPA) ---
print("Starting Phase 2: Semantic Generalization (Pure JEPA)")
dataset.pairs = [] # Flush Mock Data
dataset._load() # Load Real JSONs
for epoch in range(50):
    loss = train_epoch_jepa('jepa', dataset, MODELS, BASE_CFG, 100, 16, beta=0.0, epoch=epoch)
    wb_run.log({"Phase_2/Loss": loss})
    print(f"Phase 2 Epoch {epoch} Loss: {loss:.4f}")
    
    if epoch % 10 == 0:
        b = dataset.sample(1)
        z_t = MODELS['target_encoder']({'state': b['state'].to(DEVICE)})['latent']
        grid = sculptor.sculpt(z_t[0:1], (1,1,30,30), steps=150)
        viz_path = f"evaluation_reports/plots/sculpt_p2_{epoch}.png"
        plot_reconstruction_dashboard(b['state'][0,0], grid[0,0], z_t[0], epoch, viz_path)
        wb_run.log({f"Phase_2/Sculpt": wandb.Image(viz_path)})

# --- PHASE 3: Execution (World Model) ---
print("Starting Phase 3: World Model Relational Training")
for epoch in range(50):
    loss = train_epoch_jepa('wm', dataset, MODELS, BASE_CFG, 100, 16, epoch=epoch)
    wb_run.log({"Phase_3/Loss": loss})
    print(f"Phase 3 Epoch {epoch} Loss: {loss:.4f}")

if wb_run: wb_run.finish()